# Copying Adam Taylor's PCam5c driver to python top layer

In [1]:
from pynq import Overlay
from pynq.lib.iic import AxiIIC 
from pynq.lib.video import *
import numpy as np
from time import sleep
import PIL.Image

from pynq import MMIO

adams = Overlay("new.bit", ignore_version=True)

In [21]:
    tx_data[0] = 0x31;  tx_data[1] = 0x00;

    iic.send(iic_cam_addr, tx_data, 2)        
    print('send complete')
    sleep(0.5)
    
    status = iic.receive(iic_cam_addr, rx_data, 1)
    if hex(rx_data[0]) != '0x78':
        print("camera not detected")
        print(rx_data)        
    else:
        print("camera detected")            

send complete
camera detected


In [2]:
help(adams)

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay new.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    mipi/demosaic        : pynq.overlay.DefaultIP
    mipi/gamma_lut       : pynq.overlay.DefaultIP
    mipi/gpio_ip_reset   : pynq.lib.axigpio.AxiGPIO
    mipi/pixel_pack      : pynq.lib.video.pipeline.PixelPacker
    mipi/v_proc_sys      : pynq.overlay.DefaultIP
    axi_intc_0           : pynq.overlay.DefaultIP
    pl_rgb1_led          : pynq.lib.axigpio.AxiGPIO
    pl_rgb2_led          : pynq.lib.axigpio.AxiGPIO
    pl_rgb3_led          : pynq.lib.axigpio.AxiGPIO
    pl_user_led          : pynq.lib.axigpio.AxiGPIO
    audio/i2s_receiver_0 : pynq.overlay.DefaultIP
    shutdown_LPD         : pynq.overlay.DefaultIP
    system_management_wiz_0 : pynq.overlay.DefaultIP
    audio/i2s_transmitter_0 : pynq.overlay.DefaultIP
    audio/axi_gpio_0     : pynq.lib.axigpio.AxiGPIO
    aud

In [22]:
camera_chip_id_low()

register 300A read correct


In [23]:
help(iic)

Help on AxiIIC in module pynq.lib.iic object:

class AxiIIC(pynq.overlay.DefaultIP)
 |  AxiIIC(description)
 |  
 |  Driver for the AXI IIC controller
 |  
 |  Method resolution order:
 |      AxiIIC
 |      pynq.overlay.DefaultIP
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, description)
 |      Create a new instance of the driver
 |      
 |      Parameters
 |      ----------
 |      description : dict
 |          Entry in the ip_dict for the IP
 |  
 |  receive(self, address, data, length, option=0)
 |      Receive data from an attached IIC slave
 |      
 |      Parameters
 |      ----------
 |      address : int
 |          Address of the slave device
 |      data : bytes-like
 |          Data to receive
 |      length : int
 |          Number of bytes to receive
 |      option : int
 |          Optionally `REPEAT_START` to keep hold of the bus
 |          between transactions
 |  
 |  send(self, address, data, length, option=0)
 |      Send data t

In [2]:
'''Instantiating a branch of the Default IP Class (overlay.py file) for each ip '''

vdma = adams.mipi.axi_vdma
demosaic = adams.mipi.demosaic
gamma = adams.mipi.gamma_lut

In [3]:
'''Extracting base address and size/length of memory region from the Default_IP class '''
gamma_base_addr = gamma.mmio.base_addr
gamma_size = gamma.mmio.length

'''Intanstiating the new periphery.mmio class by feeding it the base address and size of memory allocation for gamma IP, as extracted from previous class '''
periphery = MMIO(gamma_base_addr, gamma_size)


def gamma_calc(gamma_val):
    gamma_reg = []
    i = 0
    for i in range(0,256):
        gamma_reg.append( 256*(i/256)**(1/gamma_val) )
    return gamma_reg

def count_loop(offset, gamma_corr):
    count= 0 
    while count < 0x200:
        periphery.write_mm(offset + count, int(count/2))
        print(count)
        count = count + 2

In [4]:
def demosaic_config(width, height):
    demosaic.write(0x10, width)
    demosaic.write(0x18, height)
    demosaic.write(0x28, 0x00)
    
    d_read = demosaic.read(0x00)
    print(hex(d_read))
    
    demosaic.write(0x00,0x81)
    
    d_read = demosaic.read(0x00)
    print(hex(d_read))
    return 

In [5]:
def gamma_config(width, height):
    gamma.write(0x10, width)
    gamma.write(0x18, height)
    gamma.write(0x20, 0x00)
    
    gamma_reg = gamma_calc(1.3)
    #print(gamma_reg,'\n')
    
    for i in range(0,512,2): 
        periphery.write_mm( 0x800 + i, int(gamma_reg[int(i/2)]))
        
    gamma_reg = gamma_calc(1.3)
    #print(gamma_reg,'\n')
    
    for i in range(0,512,2): 
        periphery.write_mm( 0x1000 + i, int(gamma_reg[int(i/2)])) 
        
    gamma_reg = gamma_calc(1.3)
    #print(gamma_reg,'\n')
    
    for i in range(0,512,2): 
        periphery.write_mm( 0x1800 + i, int(gamma_reg[int(i/2)]))

    gamma.write(0x00, 0x81)
    #gamma.read(0x00)
    return 

In [24]:
iic = adams.mipi.axi_iic_0

iic_cam_addr = 0x3c
sw_iic_addr = 0x3c
rx_data = bytes(10)
tx_data = [0,0,0,0,0,0]

In [25]:
def setup_i2c_switch():
    tx_data[0] = 0x04
    iic.send(sw_iic_addr, tx_data, 1)               

def detect_camera():
    tx_data[0] = 0x31;  tx_data[1] = 0x00;

    iic.send(iic_cam_addr, tx_data, 2)        
    print('send complete')
    sleep(0.5)
    
    status = iic.receive(iic_cam_addr, rx_data, 1)
    if hex(rx_data[0]) != '0x78':
        print("camera not detected")
        print(rx_data)        
    else:
        print("camera detected")            

In [9]:
def reset_camera(cfg):   
    tx_data[0] = 0x30; tx_data[1] = 0x08; tx_data[2] = cfg;
    
    iic.send(iic_cam_addr, tx_data, 3)          

In [10]:
def camera_chip_id_low():   
    tx_data[0] = 0x30;  tx_data[1] = 0x0A;

    iic.send(iic_cam_addr, tx_data, 2)               
    status = iic.receive(iic_cam_addr, rx_data, 1)

    if hex(rx_data[0]) != '0x56':
        print("Read fail at register 300A ")
    else:
        print("register 300A read correct")

In [11]:
def camera_chip_id_high():    
    tx_data[0] = 0x30;  tx_data[1] = 0x0B;

    iic.send(iic_cam_addr,tx_data,2)               
    status = iic.receive(iic_cam_addr, rx_data, 1)

    if hex(rx_data[0]) != '0x40':
        print("Read fail at register 300A ")
    else:
        print("register 300A read correct")

In [12]:
def camera_system_clk(): 
    tx_data[0] = 0x31;  tx_data[1] = 0x03;
    tx_data[2] = 0x11

    iic.send(iic_cam_addr, tx_data, 3) 

In [13]:
def iic_tx(reg, val):
    tx_data[0] = (reg>>8) & (0xFF);
    tx_data[1] = reg & (0xFF);
    tx_data[2] = val
    
    iic.send(iic_cam_addr, tx_data, 3) 
    print("sent -> {} {} {}".format(hex(tx_data[0]),hex(tx_data[1]),hex(tx_data[2])))

In [14]:
def vdma_init(videomode):
    if vdma.readchannel.running: 
        vdma.readchannel.stop()

    vdma.readchannel.mode = videomode

Start Driver Steps

In [26]:
setup_i2c_switch()

In [27]:
detect_camera()

send complete
camera detected


In [28]:
videomode = VideoMode(1920,1080,24)
                      
vdma_init(videomode)  

In [29]:
demosaic_config(1920,1080)
gamma_config(1920, 1080)

0x81
0x81


MemoryError: Unaligned write: offset must be multiple of 4.

In [35]:
%run pcam_cfg.ipynb
camera_system_clk()
iic_tx(0x3008, 0x82)
sleep(1.0)

iic_tx(0x3008, 0x42)
setting_func(cfg_init)
iic_tx(0x3008, 0x02)

iic_tx(0x3008, 0x42)
# setting_func(cfg_1080p_30fps)
setting_func(cfg_720p_60fps)
iic_tx(0x3008, 0x02)

iic_tx(0x3008, 0x42)
setting_func(cfg_simple_awb)
iic_tx(0x3008, 0x02)

vdma.readchannel.start()

sent -> 0x30 0x8 0x82
sent -> 0x30 0x8 0x42
0 / 63 0x30 0x8 0x42
sent
1 / 63 0x31 0x3 0x3
sent
2 / 63 0x30 0x17 0x0
sent
3 / 63 0x30 0x18 0x0
sent
4 / 63 0x30 0x34 0x18
sent
5 / 63 0x30 0x35 0x11
sent
6 / 63 0x30 0x36 0x38
sent
7 / 63 0x30 0x37 0x11
sent
8 / 63 0x31 0x8 0x1
sent
9 / 63 0x30 0x3d 0x10
sent
10 / 63 0x30 0x3b 0x19
sent
11 / 63 0x36 0x30 0x2e
sent
12 / 63 0x36 0x31 0xe
sent
13 / 63 0x36 0x32 0xe2
sent
14 / 63 0x36 0x33 0x23
sent
15 / 63 0x36 0x21 0xe0
sent
16 / 63 0x37 0x4 0xa0
sent
17 / 63 0x37 0x3 0x5a
sent
18 / 63 0x37 0x15 0x78
sent
19 / 63 0x37 0x17 0x1
sent
20 / 63 0x37 0xb 0x60
sent
21 / 63 0x37 0x5 0x1a
sent
22 / 63 0x39 0x5 0x2
sent
23 / 63 0x39 0x6 0x10
sent
24 / 63 0x39 0x1 0xa
sent
25 / 63 0x37 0x31 0x2
sent
26 / 63 0x36 0x0 0x37
sent
27 / 63 0x36 0x1 0x33
sent
28 / 63 0x30 0x2d 0x60
sent
29 / 63 0x36 0x20 0x52
sent
30 / 63 0x37 0x1b 0x20
sent
31 / 63 0x47 0x1c 0x50
sent
32 / 63 0x3a 0x13 0x43
sent
33 / 63 0x3a 0x18 0x0
sent
34 / 63 0x3a 0x19 0xf8
sent
35 / 63 

In [38]:
vdma.readchannel.start()
frame = vdma.readchannel.readframe()

print(len(frame))
PIL.Image.fromarray(frame[:,:,[2,1,0]])

KeyboardInterrupt: 

In [61]:
import time
num_frames = 20000000
start = time.time()

displayport = DisplayPort()
print(displayport)
displayport.configure(videomode, PIXEL_RGB)
print(displayport)
print(videomode)
for _ in range (num_frames):
    frame = displayport.newframe()
    frame[:] = vdma.readchannel.readframe()
    displayport.writeframe(frame)

end = time.time()
duration = end - start
print(f"Took {duration} seconds at {num_frames / duration} FPS")

VideoMode: width=1920 height=1080 bpp=24 fps=60


KeyboardInterrupt: 

In [ ]:
displayport.close()

vdma.readchannel.stop()

#base.free()